# Case-Based Reasoning (CBR)
## Tahap 2 - Case Representation

Tujuan:
- Mengekstrak metadata dari putusan
- Mengekstrak informasi penting
- Membuat representasi kasus terstruktur
- Menyimpan hasil ke cases.csv

1.Import Library

In [19]:
import os
import re
import pandas as pd
from datetime import datetime

2.Setup Folder

In [20]:
RAW_DIR = "../data/raw"
PROCESSED_DIR = "../data/processed"


os.makedirs(PROCESSED_DIR, exist_ok=True)

print('Folder siap digunakan')

Folder siap digunakan


3.Membaca Semua File TXT

In [21]:
text_file = sorted([
    f for f in os.listdir(RAW_DIR)
    if f.endswith(".txt")
])

print('Jumlah file:', len(text_file))

Jumlah file: 50


## i. Ekstraksi Metadata

4.Fungsi Ekstrasi Nomor Perkara

In [22]:
def extract_nomor_perkara(text):

    match = re.search(
        r'([0-9]{1,5}\s*(?:k|pk)\s*pid(?:\.sus)?\s*[0-9]{4})',
        text,
        flags=re.IGNORECASE
    )

    if match:
        hasil = match.group(1)
        hasil = re.sub(r'\s+', ' ', hasil)
        return hasil.strip()

    return "Tidak Ditemukan"

5.Fungsi Extrasi Tanggal

In [23]:
def extract_tanggal_putusan(text):

    match = re.search(
        r'demikianlah diputuskan.*?tanggal\s+'
        r'(\d{1,2}\s+'
        r'(?:januari|februari|maret|april|mei|juni|'
        r'juli|agustus|september|oktober|november|desember)'
        r'\s+\d{4})',
        text,
        flags=re.IGNORECASE | re.DOTALL
    )

    if match:
        return match.group(1)

    # fallback
    tanggal = re.findall(
        r'(\d{1,2}\s+'
        r'(?:januari|februari|maret|april|mei|juni|'
        r'juli|agustus|september|oktober|november|desember)'
        r'\s+\d{4})',
        text,
        flags=re.IGNORECASE
    )

    if tanggal:
        return tanggal[-1]

    return "Tidak Ditemukan"

Normalisasi date

In [24]:
def normalize_date(date_str):

    if pd.isna(date_str) or date_str == "Tidak Ditemukan":
        return "Tidak Ditemukan"

    formats = [
        "%d %B %Y",   # 5 maret 2024
        "%d-%b-%y",   # 11-Nov-24
        "%d-%b-%Y"    # 11-Nov-2024
    ]

    bulan_id = {
        'januari':'January',
        'februari':'February',
        'maret':'March',
        'april':'April',
        'mei':'May',
        'juni':'June',
        'juli':'July',
        'agustus':'August',
        'september':'September',
        'oktober':'October',
        'november':'November',
        'desember':'December'
    }

    temp = date_str.lower()

    for indo, eng in bulan_id.items():
        temp = temp.replace(indo, eng)

    for fmt in formats:
        try:
            return datetime.strptime(temp, fmt).strftime("%Y-%m-%d")
        except:
            pass

    return date_str

6.Fungsi Extraksi Pasal

In [25]:
def extract_pasal(text):

    pasal = re.findall(
        r'pasal\s+\d+',
        text,
        re.IGNORECASE
    )

    if pasal:
        return ", ".join(sorted(set(pasal)))
    
    return "Tidak Ditemukan"

## ii. Ektraksi Konten Kunci

7.Ringkasan Fakta

In [26]:
def extract_ringkasan_fakta(text):

    text = str(text)

    patterns = [

        r'berdasarkan fakta hukum.*?(?=menimbang bahwa berdasarkan pertimbangan tersebut)',

        r'fakta hukum yang relevan.*?(?=menimbang bahwa berdasarkan pertimbangan tersebut)',

        r'berdasarkan fakta hukum.*?(?=menimbang bahwa dengan demikian)',

        r'fakta hukum yang relevan.*?(?=menimbang bahwa dengan demikian)'

    ]

    for pattern in patterns:

        match = re.search(
            pattern,
            text,
            flags=re.IGNORECASE | re.DOTALL
        )

        if match:

            hasil = match.group(0)

            return hasil[:3000]

    # fallback
    idx = text.find("berdasarkan fakta hukum")

    if idx != -1:
        return text[idx:idx+3000]

    return text[:3000]

8.Amar Putusan

In [27]:
def extract_amar_putusan(text):

    keywords = [
        "mengadili",
        "dinyatakan ditolak",
        "menolak permohonan",
        "mengabulkan permohonan"
    ]

    for keyword in keywords:

        idx = text.find(keyword)

        if idx != -1:
            return text[idx:idx+1500]

    return "Tidak Ditemukan"

## iii. Feature Engginering

9.Fungsi Fitur

In [28]:
def calculate_features(text):

     return {
        "jumlah_kata": len(text.split()),
        "jumlah_karakter": len(text)
    }

10.Membentuk Dataset Kasus

In [29]:
cases = []

for idx, file in enumerate(text_file, start=1):

    path = os.path.join(RAW_DIR, file)

    with open(path, "r", encoding="utf-8") as f:
        text = f.read()

    features = calculate_features(text)

    cases.append({
    "case_id": idx,
    "file_name": file,
    "no_perkara": extract_nomor_perkara(text),
    "tanggal": normalize_date(
        extract_tanggal_putusan(text)
    ),
    "pasal": extract_pasal(text),
    "ringkasan_fakta": extract_ringkasan_fakta(text),
    "amar_putusan": extract_amar_putusan(text),
    "jumlah_kata": features["jumlah_kata"],
    "jumlah_karakter": features["jumlah_karakter"],
    "text_full": text
})

11.Membuat Dataframe

In [30]:
df = pd.DataFrame(cases)

df.head()

,case_id,file_name,no_perkara,tanggal,pasal,ringkasan_fakta,amar_putusan,jumlah_kata,jumlah_karakter,text_full
0,1,case_001.txt,1013 k pid.sus 2026,2025-08-07,"pasal 2, pasal 76, pasal 88",berdasarkan fakta hukum yang relevan secara yu...,mengadili telah dilaksanakan menurut ketentu...,1351,9333,gnomor 1013 k pid.sus 2026 memeriksa perkara t...
1,2,case_002.txt,1092 k pid 2024,2024-03-06,"pasal 12, pasal 2, pasal 30",gnomor 1092 k pid 2024 memeriksa perkara tinda...,dinyatakan ditolak dengan perbaikan e menimnba...,1230,8511,gnomor 1092 k pid 2024 memeriksa perkara tinda...
2,3,case_003.txt,1153 pk pid.sus 2024,2024-08-07,"pasal 2, pasal 263, pasal 266, pasal 296, pasa...",gnomor 1153 pk pid.sus 2024 memeriksa perkara ...,dinyatakan ditolak dan putusan yang dimohonkan...,882,6185,gnomor 1153 pk pid.sus 2024 memeriksa perkara ...
3,4,case_004.txt,1171 pk pid.sus 2026,2026-01-05,"pasal 2, pasal 318, pasal 322, pasal 455, pasa...",gnomor 1171 pk pid.sus 2026 memeriksa perkara ...,dinyatakan ditolak dan putusan yang dimohonkan...,943,6493,gnomor 1171 pk pid.sus 2026 memeriksa perkara ...
4,5,case_005.txt,11809 k pid.sus 2025,2025-12-11,"pasal 2, pasal 253, pasal 296",gnomor 11809 k pid.sus 2025 memeriksa perkara ...,menolak permohonan kasasi dari pemohon kasasi ...,1354,9340,gnomor 11809 k pid.sus 2025 memeriksa perkara ...


In [31]:
from sklearn.feature_extraction.text import CountVectorizer

bow = CountVectorizer(
    max_features=100,
    stop_words=None,
    token_pattern=r'(?u)\b[a-zA-Z][a-zA-Z]+\b'
)

X_bow = bow.fit_transform(
    df["text_full"]
)

print("Shape BoW:", X_bow.shape)

print("\n20 fitur teratas:")
print(bow.get_feature_names_out()[:20])

Shape BoW: (50, 100)

20 fitur teratas:
['agung' 'akta' 'alasan' 'alias' 'atas' 'atau' 'ayat' 'bahwa' 'berikut'
 'bersalah' 'bulan' 'dakwaan' 'dalam' 'dan' 'dari' 'denda' 'dengan' 'di'
 'diajukan' 'dijatuhkan']


In [32]:
bow_df = pd.DataFrame(
    X_bow.toarray(),
    columns=bow.get_feature_names_out()
)

bow_df.head()

,agung,akta,alasan,alias,atas,atau,ayat,bahwa,berikut,bersalah,...,terpidana,tersebut,tidak,tindak,tinggi,tingkat,umum,undang,untuk,yang
0,2,4,5,7,5,3,3,10,5,2,...,0,15,5,4,4,2,14,47,2,27
1,4,2,4,14,1,3,2,8,7,4,...,0,17,5,9,5,3,13,19,3,21
2,7,2,6,0,0,2,4,10,5,1,...,17,9,6,4,0,0,3,15,2,19
3,4,2,4,0,3,1,5,6,6,1,...,13,11,2,5,1,1,2,38,5,16
4,6,4,7,7,1,7,4,11,4,2,...,0,16,7,8,5,3,13,17,8,21


12.Cek Missing Metadata

In [33]:
df[
    ["no_perkara", "tanggal", "pasal"]
].head()

,no_perkara,tanggal,pasal
0,1013 k pid.sus 2026,2025-08-07,"pasal 2, pasal 76, pasal 88"
1,1092 k pid 2024,2024-03-06,"pasal 12, pasal 2, pasal 30"
2,1153 pk pid.sus 2024,2024-08-07,"pasal 2, pasal 263, pasal 266, pasal 296, pasa..."
3,1171 pk pid.sus 2026,2026-01-05,"pasal 2, pasal 318, pasal 322, pasal 455, pasa..."
4,11809 k pid.sus 2025,2025-12-11,"pasal 2, pasal 253, pasal 296"


13.Simpan Cases CSV

In [34]:
output_path = os.path.join(
    PROCESSED_DIR,
    "cases.csv"
)

df.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print("Disimpan:", output_path)

Disimpan: ../data/processed\cases.csv


In [35]:
print("="*50)
print("RINGKASAN TAHAP 2")
print("="*50)

print("Jumlah kasus:", len(df))
print("Jumlah kolom:", len(df.columns))
print("Output:", output_path)

RINGKASAN TAHAP 2
Jumlah kasus: 50
Jumlah kolom: 10
Output: ../data/processed\cases.csv


In [36]:
print(df.shape)

print(
    "No perkara kosong:",
    (df["no_perkara"] == "Tidak Ditemukan").sum()
)

print(
    "Tanggal kosong:",
    (df["tanggal"] == "Tidak Ditemukan").sum()
)

print(
    "Ringkasan kosong:",
    (df["ringkasan_fakta"] == "Tidak Ditemukan").sum()
)

print(
    "Amar kosong:",
    (df["amar_putusan"] == "Tidak Ditemukan").sum()
)

(50, 10)
No perkara kosong: 0
Tanggal kosong: 0
Ringkasan kosong: 0
Amar kosong: 2
